# Converting a model to FPGA firmware with hls4ml

A Level-1 trigger algorithm runs on an FPGA and must decide in well under a microsecond. [hls4ml](https://fastmachinelearning.org/hls4ml/)
translates a trained neural network into C++ for *high-level synthesis* (HLS): AMD/Xilinx tools then turn that C++ into FPGA firmware.

Two things change when a model moves to the FPGA:

- **Fixed-point arithmetic.** Instead of 32-bit floats, every number is an `ap_fixed<W, I>`: `W` bits in total, `I` of them
  for the integer part (including the sign), `W - I` for the fraction. Fewer bits = fewer resources and lower latency, but
  less precision, so the model's output changes slightly.
- **Parallelism.** `ReuseFactor = 1` means every multiplication gets its own multiplier: fastest, but the most resources.
  A larger reuse factor shares multipliers between operations, trading latency for resources.

In this notebook you will:

1. build an hls4ml **configuration**, which says which precision every layer uses,
2. **convert** a trained PyTorch model to C++ and **compile** it, so it can run on a CPU exactly as it would on the FPGA,
3. **compare** its anomaly scores with the PyTorch model, and find out **why** they differ, layer by layer,
4. **time** the two,
5. **scan the bit width** to see how many bits the model needs,
6. convert a model trained with **quantization-aware training** (Brevitas), which knows its bit widths from training.
7. **evaluate the firmware**: score the C++ models, check that the trigger threshold still gives the right rate, and compare them all in one plot.

Cells with `___` are for you to fill in: replace every `___` before running the cell.

In [ ]:
import copy
import time
import warnings
from pathlib import Path

import hls4ml
import matplotlib.pyplot as plt
import numpy as np
import torch

import plotting
from dataloader import EventData, create_datasets
from models.autoencoder import predict, reconstruction_error
from utils import (compute_scores, efficiency_at_acceptance, load_inputs, load_model, load_trials, model_cost, roc,
                   save_trial, summary, threshold_at_acceptance)


class _Blank:
    """Placeholder for the ___ to fill in. Without it, Jupyter's ___ is the output of an earlier cell."""

    def __getattr__(self, name):
        raise RuntimeError("this cell still has a ___ to fill in")

    def __repr__(self):
        return "___ (fill this in)"


___ = _Blank()

In [ ]:
MODEL_PATH = "output/dense-ae/best_model.pt"  # the model to convert (a model without quantized layers, see section 6 for those)
CONFIG = "config/baseline.yml"
ACCEPTANCE = 1e-3

# The C++ model is much slower than PyTorch (see section 4): evaluate on (at most) this many events per sample.
N_EVENTS = 50_000

# FPGA settings
BACKEND = "Vitis"                # the HLS tool the C++ is written for: "Vitis" or "Vivado"
PART = "xcvu13p-flga2577-2-e"    # a Virtex UltraScale+ FPGA, the family used in the CMS Level-1 trigger
CLOCK_PERIOD = 6.25              # ns
OUTPUT_DIR = "output/hls4ml/" + MODEL_PATH.split("/")[-2]

model = load_model(MODEL_PATH)

In [ ]:
def load_test(model_path):
    """The test datasets, with the inputs the model was trained with, and at most N_EVENTS events per sample."""
    inputs = load_inputs(model_path)  # the layout, features and input type the model was trained with
    _, _, test = create_datasets(CONFIG, layout=inputs["layout"], features=inputs["features"],
                                 input_type=inputs["input_type"], load_test=True, load_train=False)
    return {name: EventData(d.x[:N_EVENTS], d.mask[:N_EVENTS]) for name, d in test.items()}


test = load_test(MODEL_PATH)
INPUT_SHAPE = test["background"].x.shape[1:]  # one event, without the batch dimension
print({name: len(d) for name, d in test.items()})
summary(model, INPUT_SHAPE)

## 1. The hls4ml configuration

hls4ml needs to know which fixed-point type to use for every number in the network. That is what the configuration is: a
nested dictionary that you can print and edit before converting.

`config_from_pytorch_model` traces the model with the shape of one event and creates the configuration:

- `config["Model"]` holds the defaults for the whole model: the `Precision` (`default_precision`), the `ReuseFactor`, and the
  `Strategy` (`"Latency"`: everything in parallel, for small models; `"Resource"`: share multipliers, for big ones).
- `config["LayerName"]` has one entry per layer (because of `granularity="name"`). The layers are named after their position
  in the `nn.Sequential`: `_0` is the first layer, `_1` the second, and so on. `input_1` is the input.
- Each layer has precisions for its `weight`s, `bias`es, `result` (its output) and `accum`ulator (the running sum inside a dense layer).
  Activations computed with a lookup table also have a `table` precision. `"auto"` means hls4ml chooses: for a dense layer it
  makes the accumulator and result wide enough that the sum can never overflow.

A precision is written `ap_fixed<W, I, Q, O>`:

- `W` total bits and `I` integer bits (including the sign). `ap_fixed<16,6>` covers -32 to 32 in steps of 2<sup>-10</sup> ≈ 0.001.
- `Q`, the *quantization mode*: what happens to bits below the last fractional bit. The default `AP_TRN` *truncates*, i.e.
  drops them, which always rounds down. `AP_RND_CONV` rounds to the nearest value.
- `O`, the *overflow mode*: what happens to a value outside the range. The default `AP_WRAP` wraps around, like integer
  overflow (with `ap_fixed<16,6>`, 33 becomes -31!). `AP_SAT` *saturates*: it clips to the largest value instead.

Rounding and saturation cost a little extra logic, which is why they are not the default.

In [ ]:
config = hls4ml.utils.config_from_pytorch_model(
    model, INPUT_SHAPE, granularity="name", backend=BACKEND, default_precision="ap_fixed<16,6>", default_reuse_factor=1,
)
config["Model"]["Strategy"] = "Latency"

print("Model:", config["Model"])
for layer, layer_cfg in config["LayerName"].items():
    print(f"{layer:8s} {layer_cfg['Precision']}")

In [ ]:
# You can change any entry before converting. For example, to give the inputs more fractional bits:
# config["LayerName"]["input_1"]["Precision"]["result"] = "ap_fixed<18,6>"
config["LayerName"]["_0"]

## 2. Converting and compiling

`convert_from_pytorch_model` turns the PyTorch model and the configuration into an hls4ml model. Besides the configuration, it needs:

- `output_dir`: where to write the C++ project,
- `backend` and `part`: which HLS tool and which FPGA the C++ is written for,
- `clock_period`: the FPGA clock, in ns,
- `io_type`: `"io_parallel"` passes all values of a layer at once (fastest, for small models), `"io_stream"` one at a time.

`hls_model.compile()` then writes the project to `output_dir` and compiles its C++ with an ordinary C++ compiler into a
library that Python can call: `hls_model.predict(x)` runs the C++ model on the CPU. It uses the same fixed-point types
the FPGA would use, so it gives exactly the same numbers as the firmware would (it is *bit-accurate*).

In [ ]:
hls_model = hls4ml.converters.convert_from_pytorch_model(
    model, hls_config=config, output_dir=OUTPUT_DIR, backend=BACKEND, part=PART,
    clock_period=CLOCK_PERIOD, io_type="io_parallel",
)
hls_model.compile()

During the conversion hls4ml filled in every `"auto"` precision. These are the types the C++ model really uses:

In [ ]:
def print_precisions(hls_model):
    """The fixed-point type of the output, weights and biases of every layer of an hls4ml model."""
    for layer in hls_model.get_layers():
        weights = {name: str(w.type.precision) for name, w in layer.weights.items()}
        print(f"{layer.name:12s} {layer.class_name:12s} output: {str(layer.get_output_variable().type.precision):26s} {weights}")


print_precisions(hls_model)

Look at the outputs of the dense layers: far more than 16 bits! Every product of a 16-bit input and a 16-bit weight has 32 bits,
and adding up `n` of them needs about log<sub>2</sub>(n) more. That is exact, but expensive on an FPGA: the activation after
each dense layer cuts the values back to 16 bits. You could set smaller `accum` and `result` precisions yourself.

### What hls4ml wrote

The C++ project in `OUTPUT_DIR` is a complete HLS project:

- `firmware/myproject.cpp`: the network, as one function call per layer,
- `firmware/parameters.h`: the configuration of every layer (sizes, reuse factor, ...), `firmware/defines.h`: the fixed-point types,
- `firmware/weights/`: the weights and biases, as C++ arrays,
- `firmware/nnet_utils/`: hls4ml's C++ library with the implementation of every layer type,
- `myproject_test.cpp` and `tb_data/`: a test bench, `build_prj.tcl`: the script that runs the HLS tool (see the last section).

In [ ]:
print(sorted(p.name for p in Path(OUTPUT_DIR).iterdir()))
print(sorted(p.name for p in Path(OUTPUT_DIR, "firmware").iterdir()))

network = Path(OUTPUT_DIR, "firmware", "myproject.cpp").read_text()
print(network[network.index("void myproject"):])

Every layer is a call to a function template from `nnet_utils` (`nnet::dense`, `nnet::relu`, ...), with the types and
sizes from `parameters.h` as template arguments. The `#pragma HLS` lines are ignored by an ordinary compiler: they tell the HLS
tool how to build the circuit (see the last section).

## 3. Does the C++ model give the same answer?

`hls_model.predict` runs the C++ model, so we can compute the anomaly score exactly as before and compare it with the PyTorch model.

Note that here only the autoencoder runs "on the FPGA": the reconstruction error is still computed in Python.
In a real trigger, the score (and the threshold cut) would also be computed in firmware.

In [ ]:
def hls_scores(hls_model, data):
    """Anomaly score of every event in an EventData, computed with the C++ model."""
    x_hat = hls_model.predict(np.ascontiguousarray(data.x))
    return reconstruction_error(data.x, data.mask, x_hat)


def summarize(scores):
    """{signal: (AUC, efficiency at ACCEPTANCE)}"""
    bkg = scores["background"]
    return {name: (roc(bkg, s)[2], efficiency_at_acceptance(bkg, s, ACCEPTANCE))
            for name, s in scores.items() if name != "background"}


def print_comparison(torch_metrics, hls_metrics):
    print(f"{'signal':35s} {'AUC torch':>10s} {'AUC hls4ml':>10s} {'eff torch':>10s} {'eff hls4ml':>10s}")
    for name in torch_metrics:
        (auc_t, eff_t), (auc_h, eff_h) = torch_metrics[name], hls_metrics[name]
        print(f"{name:35s} {auc_t:10.4f} {auc_h:10.4f} {eff_t:10.4f} {eff_h:10.4f}")


torch_scores = {name: compute_scores(model, d) for name, d in test.items()}
hls_model_scores = {name: hls_scores(hls_model, d) for name, d in test.items()}
torch_metrics, hls_metrics = summarize(torch_scores), summarize(hls_model_scores)
print_comparison(torch_metrics, hls_metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
k, h = torch_scores["background"], hls_model_scores["background"]
ax.scatter(k, h, s=2, alpha=0.3)
lim = [0, np.quantile(k, 0.999)]
ax.plot(lim, lim, color="gray", linestyle="--", linewidth=1)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("anomaly score (PyTorch, float)")
ax.set_ylabel("anomaly score (hls4ml, fixed point)")
ax.set_title("zero-bias test events");

### Why are the C++ and PyTorch models different?

PyTorch computes with 32-bit floating-point numbers: about 7 significant digits, over a huge range. The C++ model computes
with the fixed-point types from the configuration, and every time a value is stored in a fixed-point type it can change:

1. **The inputs are rounded** to the precision of `input_1`. With `ap_fixed<16,6>`, to a multiple of 2<sup>-10</sup> ≈ 0.001.
2. **The weights and biases are rounded** in the same way.
3. **The output of every activation is rounded again** to 16 bits (the dense layers themselves are exact here, thanks to their
   wide `"auto"` accumulators). The error of one layer is then passed on to the next, multiplied by its weights, so it
   grows through the network.
4. **Truncation rounds down.** With the default `AP_TRN`, every rounding goes in the same direction, so the errors don't
   average out: the C++ outputs are systematically a bit *smaller* than PyTorch's.
5. **Overflow wraps around.** A value outside the range of its type (±32 for `ap_fixed<16,6>`) wraps around to the other end
   with the default `AP_WRAP`. This is rare, but when it happens the output is completely wrong.
6. **Lookup tables.** Activations such as sigmoid or tanh are read from a table with `TableSize` entries instead of being
   computed. (ReLU, used here, is computed exactly.)

None of this is a bug: the C++ model computes exactly what the FPGA would. The question is whether the differences matter for
the anomaly score.

hls4ml can show you where the differences come from. With `"Trace": True` for a layer, `hls_model.trace(x)` returns the
output of that layer for every event, as well as the final output. Below we compare them with the output of every PyTorch
layer, recorded with *forward hooks* (functions PyTorch calls after a layer has run).

In [ ]:
x = np.ascontiguousarray(test["background"].x[:5000])

# hls4ml: switch on tracing for every layer, convert again, and record the output of every layer
trace_config = copy.deepcopy(config)
for layer_cfg in trace_config["LayerName"].values():
    layer_cfg["Trace"] = True
trace_model = hls4ml.converters.convert_from_pytorch_model(
    model, hls_config=trace_config, output_dir=OUTPUT_DIR + "_trace", backend=BACKEND, part=PART,
    clock_period=CLOCK_PERIOD, io_type="io_parallel",
)
trace_model.compile()
_, hls_trace = trace_model.trace(x)  # {layer name: output of that layer}

# PyTorch: record the output of every layer with forward hooks. Layer "0" in PyTorch is "_0" in hls4ml.
torch_trace = {}
hooks = [layer.register_forward_hook(lambda module, inputs, output, name=name: torch_trace.update({f"_{name}": output.numpy()}))
         for name, layer in model.named_children()]
with torch.no_grad():
    model(torch.as_tensor(x))
for hook in hooks:
    hook.remove()

print(f"{'layer':6s} {'type':8s} {'max |PyTorch|':>14s} {'mean diff':>10s} {'max |diff|':>10s}")
for name, hls_out in hls_trace.items():
    diff = hls_out - torch_trace[name].reshape(hls_out.shape)
    kind = type(model[int(name[1:])]).__name__
    print(f"{name:6s} {kind:8s} {np.abs(torch_trace[name]).max():14.3f} {diff.mean():10.4f} {np.abs(diff).max():10.4f}")

How does the difference grow through the network? Is the mean difference (C++ minus PyTorch) positive or negative, and why?
Are any values close to the edge of the range of `ap_fixed<16,6>`?

#### Try it: round instead of truncating

Convert the model again with rounding to the nearest value (`AP_RND_CONV`) and saturation (`AP_SAT`) for every layer, keeping
16 bits of which 6 are integer bits. Fill in the precision. `convert` below does the two steps of section 2 in one go, and
`output_difference` summarises how far the C++ output is from the PyTorch output.

In [ ]:
def convert(model, config, output_dir):
    """Convert a PyTorch model with an hls4ml configuration, and compile it so that .predict works."""
    hls_model = hls4ml.converters.convert_from_pytorch_model(
        model, hls_config=config, output_dir=output_dir, backend=BACKEND, part=PART,
        clock_period=CLOCK_PERIOD, io_type="io_parallel",
    )
    hls_model.compile()
    return hls_model


def output_difference(torch_model, hls_model, x):
    """How far the output of the C++ model is from the PyTorch model, for the events x."""
    diff = hls_model.predict(np.ascontiguousarray(x)) - predict(torch_model, x)
    return {"mean diff": float(diff.mean()), "mean |diff|": float(np.abs(diff).mean()), "max |diff|": float(np.abs(diff).max())}


rounding_config = hls4ml.utils.config_from_pytorch_model(
    model, INPUT_SHAPE, granularity="name", backend=BACKEND, default_reuse_factor=1,
    default_precision=___,
)
rounding_model = convert(model, rounding_config, OUTPUT_DIR + "_rounding")

print("truncate, wrap:   ", output_difference(model, hls_model, x))
print("round, saturate:  ", output_difference(model, rounding_model, x))
print_comparison(torch_metrics, summarize({name: hls_scores(rounding_model, d) for name, d in test.items()}))

## 4. How fast is it?

Let's time the two models on the same events.

In [ ]:
x_timing = np.ascontiguousarray(test["background"].x)

start = time.perf_counter()
predict(model, x_timing)
torch_time = time.perf_counter() - start

start = time.perf_counter()
hls_model.predict(x_timing)
hls_time = time.perf_counter() - start

n = len(x_timing)
print(f"{n:,} events")
print(f"PyTorch:      {torch_time:7.2f} s = {torch_time / n * 1e6:8.2f} µs per event")
print(f"C++ (hls4ml): {hls_time:7.2f} s = {hls_time / n * 1e6:8.2f} µs per event   ({hls_time / torch_time:.0f}x slower)")

The C++ model is much *slower* than PyTorch, although it is the "firmware". That is because it *emulates* the FPGA on a CPU:

- Every `ap_fixed` number is a C++ class, and every addition or multiplication takes many CPU instructions to reproduce the
  fixed-point arithmetic bit by bit. A CPU is built for floating-point numbers, and PyTorch uses them directly.
- The C++ model processes one event at a time, one multiplication after the other. PyTorch processes thousands of events at
  once, with matrix multiplications optimized for the CPU's vector instructions.

Neither number tells you how fast the model is on the FPGA. There, with `ReuseFactor = 1` and the `Latency` strategy, all
multiplications of a layer happen at the same time, in dedicated hardware. A small network like this one typically takes a few
clock cycles per layer: a latency of the order of 100 ns at a clock period of 6.25 ns. And because the circuit is pipelined,
it can start on a new event every clock cycle, while the LHC delivers one every 25 ns. Only synthesis (last section)
gives the exact numbers.

## 5. How many bits do we need?

Fewer bits make the model cheaper on the FPGA. How far can we go? Convert the model with fewer and fewer total bits `W` in
`ap_fixed<W, 6>` (keeping 6 integer bits), and for each, record:

- the mean AUC over the signals, as in section 3,
- the mean |difference| between the C++ and PyTorch outputs (`output_difference(...)["mean |diff|"]`).

Fill in the blanks. The first is the precision for `bits` total bits and `INTEGER_BITS` integer bits.

In [ ]:
TOTAL_BITS = [8, 10, 12, 14, 16, 18]
INTEGER_BITS = 6

mean_auc, mean_diff = [], []
for bits in TOTAL_BITS:
    precision = f"ap_fixed<___>"
    scan_config = hls4ml.utils.config_from_pytorch_model(
        model, INPUT_SHAPE, granularity="name", backend=BACKEND, default_reuse_factor=1, default_precision=___,
    )
    scan_model = convert(model, scan_config, f"{OUTPUT_DIR}_{bits}bits")

    scan_scores = {name: hls_scores(___, d) for name, d in test.items()}
    mean_auc.append(np.mean([auc for auc, _ in summarize(___).values()]))
    mean_diff.append(output_difference(model, ___, x)["mean |diff|"])
    print(f"{precision:16s} mean AUC = {mean_auc[-1]:.4f}   mean |diff| = {mean_diff[-1]:.4f}")

In [ ]:
fig, (ax_auc, ax_diff) = plt.subplots(1, 2, figsize=(11, 4))
ax_auc.plot(TOTAL_BITS, mean_auc, marker="o", label="hls4ml (fixed point)")
ax_auc.axhline(np.mean([auc for auc, _ in torch_metrics.values()]), color="gray", linestyle="--", label="PyTorch (float)")
ax_auc.set_ylabel("mean AUC over signals")
ax_auc.legend()
ax_diff.plot(TOTAL_BITS, mean_diff, marker="o")
ax_diff.set_yscale("log")
ax_diff.set_ylabel("mean |C++ output - PyTorch output|")
for ax in (ax_auc, ax_diff):
    ax.set_xlabel(f"total bits W in ap_fixed<W,{INTEGER_BITS}>")
fig.tight_layout()

- How many bits does the model need before the C++ model behaves like the PyTorch model?
- Does the AUC always go down when the C++ model moves away from the PyTorch model? Why might it not?
- Repeat the scan with `AP_RND_CONV,AP_SAT`. Does it change how many bits you need?
- Not every layer needs the same precision. Give one layer more bits (e.g. `scan_config["LayerName"]["input_1"]["Precision"]["result"]`)
  and see which layers matter most.

With few bits the fixed-point model is effectively a *different* model from the one you trained, and nothing in training
prepared it for that. *Quantization-aware training* addresses this by simulating the fixed-point arithmetic during training.

## 6. Quantization-aware training: converting a Brevitas model

A model trained with quantization-aware training already knows its bit widths: `dense-qae` (in
[`models/autoencoder.py`](models/autoencoder.py)) rounds its weights and activations to 6 bits during training, so there is
nothing for hls4ml to guess. hls4ml does not read Brevitas layers directly (`config_from_pytorch_model` fails on them). Instead,
the model goes through [QONNX](https://github.com/fastmachinelearning/qonnx), an extension of the ONNX format for neural networks
with a `Quant` node for every quantizer, which records its bit width:

1. Brevitas exports the model to a QONNX file.
2. The QONNX tools clean up the graph into the form hls4ml expects.
3. hls4ml reads the QONNX graph (`config_from_onnx_model`, `convert_from_onnx_model`) and gives every layer exactly the
   precision it was trained with. `default_precision` only applies to what was not quantized, like the output of the last layer.

Train `dense-qae` in [`train.ipynb`](train.ipynb) first (set `model_type = "dense-qae"`). To convert your own quantized model,
build it from `quant_input`, `quant_dense` and `quant_relu` (also in `models/autoencoder.py`): hls4ml needs the
power-of-two scales and quantized biases they use.

In [ ]:
QAT_MODEL_PATH = "output/dense-qae/best_model.pt"  # a model with quantized (Brevitas) layers, trained with train.ipynb
QAT_OUTPUT_DIR = "output/hls4ml/" + QAT_MODEL_PATH.split("/")[-2]

qat_model = load_model(QAT_MODEL_PATH)
qat_test = load_test(QAT_MODEL_PATH)
QAT_INPUT_SHAPE = qat_test["background"].x.shape[1:]
summary(qat_model, QAT_INPUT_SHAPE)

In [ ]:
from brevitas.export import export_qonnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.channels_last import ConvertToChannelsLastAndClean
from qonnx.transformation.gemm_to_matmul import GemmToMatMul
from qonnx.util.cleanup import cleanup_model

onnx_path = Path(f"{QAT_OUTPUT_DIR}.onnx")
onnx_path.parent.mkdir(parents=True, exist_ok=True)

# 1. Export: Brevitas runs the model on one (empty) event and records the graph.
#    dynamo=False: the older TorchScript exporter gives a graph that hls4ml understands.
export_qonnx(copy.deepcopy(qat_model).eval(), torch.zeros(1, *QAT_INPUT_SHAPE), export_path=str(onnx_path), dynamo=False)

# 2. Clean up: give every tensor a name and a shape, and fold away constant computations
qonnx_model = cleanup_model(ModelWrapper(str(onnx_path)))
qonnx_model = qonnx_model.transform(ConvertToChannelsLastAndClean())  # hls4ml stores data channels-last
with warnings.catch_warnings():  # GemmToMatMul warns about the ONNX version, but works
    warnings.filterwarnings("ignore", message=".*GemmToMatMul.*")
    qonnx_model = qonnx_model.transform(GemmToMatMul())  # y = Wx + b (Gemm) -> MatMul + Add, which hls4ml converts
qonnx_model = cleanup_model(qonnx_model)

# The graph: Quant nodes (the quantizers), MatMul + Add (the dense layers) and Relu
print([node.op_type for node in qonnx_model.graph.node])

In [ ]:
# 3. Convert with hls4ml
qat_config = hls4ml.utils.config_from_onnx_model(
    qonnx_model, granularity="name", backend=BACKEND, default_precision="ap_fixed<16,6>", default_reuse_factor=1,
)
qat_hls_model = hls4ml.converters.convert_from_onnx_model(
    qonnx_model, hls_config=qat_config, output_dir=QAT_OUTPUT_DIR, backend=BACKEND, part=PART,
    clock_period=CLOCK_PERIOD, io_type="io_parallel",
)
qat_hls_model.compile()
print_precisions(qat_hls_model)

In [ ]:
qat_torch_metrics = summarize({name: compute_scores(qat_model, d) for name, d in qat_test.items()})
qat_hls_metrics = summarize({name: hls_scores(qat_hls_model, d) for name, d in qat_test.items()})
print_comparison(qat_torch_metrics, qat_hls_metrics)
print(output_difference(qat_model, qat_hls_model, qat_test["background"].x[:5000]))

How close are the PyTorch and hls4ml numbers now, compared with the float model converted at a similar number of bits
in the scan above? Compare the cost of the two models too (`model_cost` in `utils.py`, or the trials plot in
[`evaluate.ipynb`](evaluate.ipynb)). Then try fewer bits: register a version of `dense-qae` with `bits=4`, train and convert it.

## 7. Evaluating the firmware

What runs in the trigger is not the PyTorch model: it is the firmware. A model that is excellent in PyTorch but falls apart
in fixed point is useless, and a model that is a bit worse in PyTorch but survives the conversion with few bits can be the
better trigger. So from here on we score the **firmware**: the C++ models, which give exactly the numbers the FPGA would.

The firmware also raises a question PyTorch can't answer: **does the threshold still give the right rate?** The trigger
does not get to choose its threshold event by event. It is fixed in advance so that the algorithm keeps `ACCEPTANCE` of the
zero-bias events, because that rate is what the trigger's bandwidth budget is about. If the firmware's scores are shifted
(remember the truncation in section 3), a threshold chosen on the PyTorch scores keeps a different fraction of events.
At 40 MHz, every 10<sup>-4</sup> of acceptance is 4 kHz of trigger rate.

For every firmware model, we compute per signal:

- `auc` and `efficiency`: as in [`evaluate.ipynb`](evaluate.ipynb), but with the firmware scores. The threshold is set on
  the firmware's own zero-bias scores, as it would be when the firmware is deployed.
- `acceptance_torch_threshold`: the fraction of zero-bias events the firmware keeps with the threshold set on the
  **PyTorch** scores (the same for all signals). Ideally this is `ACCEPTANCE`.
- `overlap`: of the signal events the PyTorch model keeps, the fraction the firmware keeps too (each with its own
  threshold). 1 means the firmware selects the same events as the model you trained.

The trials are saved in their own file, `output/firmware_trials.csv`, so they don't mix with the PyTorch trials of `evaluate.ipynb`.

A word on statistics: with `N_EVENTS` = 50,000 zero-bias events and `ACCEPTANCE` = 10<sup>-3</sup>, the threshold is set by
the 50 highest-scoring events, so a measured acceptance has a statistical uncertainty of about 1/&radic;50 &approx; 14%.
Smaller shifts are noise.

In [ ]:
FIRMWARE_TRIALS = "output/firmware_trials.csv"

# The firmware models to evaluate: (trial name, PyTorch model, hls4ml model, test data, model path, bits).
# bits: the bit width model_cost assumes for the layers that were not quantized with Brevitas, i.e. W in ap_fixed<W,I>.
# Add your own models here, e.g. a model from the bit-width scan of section 5 (convert it again with `convert`).
FIRMWARE = [
    ("dense-ae <16,6>", model, hls_model, test, MODEL_PATH, 16),
    ("dense-ae <16,6> round", model, rounding_model, test, MODEL_PATH, 16),
    ("dense-qae", qat_model, qat_hls_model, qat_test, QAT_MODEL_PATH, 16),
]

In [ ]:
def evaluate_firmware(torch_model, hls_model, data):
    """Metrics of the firmware for every signal ({signal: {metric: value}}), and the firmware scores of every sample."""
    fw = {name: hls_scores(hls_model, d) for name, d in data.items()}
    pt = {name: compute_scores(torch_model, d) for name, d in data.items()}
    fw_threshold = threshold_at_acceptance(fw["background"], ACCEPTANCE)
    pt_threshold = threshold_at_acceptance(pt["background"], ACCEPTANCE)

    # Rate stability: the fraction of zero-bias events the firmware keeps with the threshold set on the PyTorch scores
    acceptance_torch_threshold = float(np.mean(___ > ___))

    metrics = {}
    for name in fw:
        if name == "background":
            continue
        kept_by_torch = pt[name] > pt_threshold
        metrics[name] = {
            "auc": roc(fw["background"], fw[name])[2],
            "efficiency": efficiency_at_acceptance(fw["background"], fw[name], ACCEPTANCE),
            "acceptance_torch_threshold": acceptance_torch_threshold,
            "overlap": float(np.mean(fw[name][kept_by_torch] > fw_threshold)) if kept_by_torch.any() else np.nan,
        }
    return metrics, fw


fw_metrics, fw_scores = {}, {}
for trial, torch_model, fw_model, data, model_path, bits in FIRMWARE:
    fw_metrics[trial], fw_scores[trial] = evaluate_firmware(torch_model, fw_model, data)
    cost = model_cost(torch_model, data["background"].x.shape[1:], default_bits=bits)
    save_trial(trial, fw_metrics[trial], cost, model_path=model_path, config=CONFIG, acceptance=ACCEPTANCE,
               notes="hls4ml firmware", path=FIRMWARE_TRIALS)

In [ ]:
fw_trials = load_trials(FIRMWARE_TRIALS)
(fw_trials.groupby("trial")
 .agg(bops=("bops", "first"), mean_auc=("auc", "mean"), mean_efficiency=("efficiency", "mean"),
      acceptance_torch_threshold=("acceptance_torch_threshold", "first"), mean_overlap=("overlap", "mean"),
      notes=("notes", "first"))
 .sort_values("bops"))

### The firmware's anomaly scores and ROC curves

The same plots as in `evaluate.ipynb`, from the firmware scores only.

In [ ]:
for trial, s in fw_scores.items():
    plotting.plot_score_distributions(s, title=f"{trial} (firmware)")
    rocs = {name: roc(s["background"], s[name]) for name in s if name != "background"}
    plotting.plot_roc(rocs, ACCEPTANCE, title=f"{trial} (firmware)")

In [ ]:
plotting.plot_signal_efficiency(
    {trial: {signal: m["efficiency"] for signal, m in metrics.items()} for trial, metrics in fw_metrics.items()},
    ACCEPTANCE,
);

### Does the threshold give the right rate?

The zero-bias acceptance of every firmware model with the threshold set on the PyTorch scores, divided by the target
`ACCEPTANCE`: 1.1 means the firmware fires 10% more often than it should. The error bars are the
statistical uncertainty: anything within them of the dashed line is consistent with the target rate.

In [ ]:
trials = list(fw_metrics)
rates = np.array([next(iter(fw_metrics[t].values()))["acceptance_torch_threshold"] for t in trials])
errors = np.sqrt(rates / np.array([len(fw_scores[t]["background"]) for t in trials]))  # Poisson uncertainty

fig, ax = plt.subplots(figsize=(7, 0.6 * len(trials) + 1.5))
ax.errorbar(rates / ACCEPTANCE, np.arange(len(trials)), xerr=errors / ACCEPTANCE, fmt="o", capsize=3)
ax.axvline(1, color="gray", linestyle="--", label=f"target: {ACCEPTANCE:g}")
ax.set_yticks(np.arange(len(trials)))
ax.set_yticklabels(trials)
ax.set_xlabel("zero-bias acceptance of the firmware with the PyTorch threshold / target")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()

### All firmware trials: performance vs. model cost

The Pareto plot of `evaluate.ipynb`, for the firmware trials only.

In [ ]:
plotting.plot_trials(load_trials(FIRMWARE_TRIALS), outpath="output/firmware_trials.png");

- Which firmware model is on the Pareto front? Is it the same one that is on the front of the PyTorch trials in `evaluate.ipynb`?
- Which models keep the target rate with the PyTorch threshold, and which shift it? In which direction? Connect this to
  section 3: the anomaly score is the reconstruction error, the difference between the input and the output. If every
  output of the decoder is a little too small, does the score go up or down?
- How would you fix a rate shift in the real trigger? (Hint: this is one reason why a bit-accurate emulation of the firmware is so useful.)
- Add a float model converted with 8 or 10 bits from the scan in section 5 to `FIRMWARE`, and compare it with `dense-qae`.
  Which is the better trigger, and which has the more stable rate?
- Is `overlap` a good measure of how faithful the firmware is? Can a model have a high AUC and a low overlap?

## Next step: synthesis

So far the "firmware" has only run on a CPU. The next step is **synthesis**: an HLS tool (AMD Vitis HLS or Vivado HLS) turns the
C++ that hls4ml wrote into a circuit, a description of the logic, multipliers and wires on the FPGA. hls4ml writes the
script for it (`build_prj.tcl`); `hls_model.build()` runs it.

This is not a one-to-one translation. A CPU runs one instruction after the other, while an FPGA can do thousands of things at
once, and the C++ alone does not say which of the many possible circuits to build. The `#pragma HLS` lines in hls4ml's C++
tell the HLS tool, for example, to *unroll* loops (one multiplier per weight instead of one multiplier used many times), to keep
arrays in registers so that all their elements can be used in the same clock cycle, or to share multipliers according to the
`ReuseFactor`. This is why the cost of a layer grows with the number of multiplications and with their bit widths, as the BOPs
in `utils.model_cost` assume.

Synthesis reports what BOPs and the number of layers only estimate: the **latency** (in clock cycles), the **initiation
interval** (how often a new event can come in), and the **resources** the model uses (DSPs, the hard-wired multipliers; LUTs
and flip-flops, the general-purpose logic; BRAM, the memory blocks). These numbers decide whether a model can run in the
trigger, next to all the other trigger algorithms on the same FPGA. Synthesis needs the AMD tools, which we don't have for this
hackathon, and takes minutes to hours per model.